# 🧬 Protein–Ligand MD + MM-GBSA on Google Colab

End-to-end demo of the [`openmm_ligands_json`](https://github.com/jinichlab/openmm_ligands_json)
pipeline. We take a **protein–ligand complex** and, step by step:

1. **split** it into protein + ligand (pH-aware ligand protonation)
2. **fix** the protein (missing atoms, hydrogens at pH)
3. **parameterize once** (GAFF / AM1-BCC via OpenFF) and **solvate**
4. minimize → **NVT** → **NPT** → **production MD**
5. analyze: **ligand RMSD**, **protein RMSF**, thermodynamics
6. compute an **MM-GBSA** binding free energy with AmberTools

With **3D views** at every stage (ligand, protein, complex, solvated box, trajectory)
and postprocessing **plots**.

> ⚙️ **Use a GPU runtime if you can:** *Runtime ▸ Change runtime type ▸ T4 GPU.*
> Everything also runs on CPU, just slower.

> This mirrors the standalone-script test the pipeline was validated with — the
> "normal" way to run at scale is the JSON + `step_runner.py` pipeline.

## 0 · Environment setup

Colab has no conda by default. We bootstrap **Mambaforge** with `condacolab`
(the kernel restarts once — that's normal), then create a dedicated
**`openmm-ligands`** conda env from the repo's `environment.yml`. **The pipeline
runs inside that env** (via its own Python); the notebook **kernel** only needs a
light visualization/analysis stack (py3Dmol, mdtraj, rdkit, plotting), which we
`pip`-install into the kernel separately.

In [ ]:
# Bootstrap Mambaforge into Colab. THE KERNEL WILL RESTART after this cell — expected.
!pip install -q condacolab
import condacolab
condacolab.install_mambaforge()

### ⏳ The kernel restarted — wait for it to reconnect, then run the cells below.
**Do not re-run the cell above.** The next cell creates the conda env from
`environment.yml` and takes ~10–15 min (ambertools is large).

In [ ]:
# (after restart) Fetch the pipeline and build its conda env from environment.yml.
import os
BRANCH = "ligand-mmpbsa-fixes"   # branch with the MM-PBSA fixes + this notebook
if not os.path.isdir("openmm_ligands_json"):
    !git clone -q -b {BRANCH} https://github.com/jinichlab/openmm_ligands_json.git

# Creates /usr/local/envs/openmm-ligands (name comes from environment.yml). ~10-15 min.
!mamba env create -f openmm_ligands_json/environment.yml

ENV = "/usr/local/envs/openmm-ligands"
# setuptools>=81 removed pkg_resources, which some deps still import at runtime.
!{ENV}/bin/python -m pip install -q "setuptools<81"
# (add any extra pip-only deps here, e.g. !{ENV}/bin/python -m pip install <pkg>)

# Put the env's bin on PATH so pipeline subprocesses find antechamber / sqm /
# MMPBSA.py — OpenFF needs antechamber for the AM1-BCC charge fit. Without this,
# calling the env python by absolute path leaves its bin off PATH and charging fails.
os.environ["PATH"] = ENV + "/bin:" + os.environ["PATH"]
os.environ["LIGAND_LOG_LEVEL"] = "INFO"   # timestamped step logs
print("pipeline env ready at", ENV)

In [ ]:
# The notebook KERNEL renders the 3D views and plots, so install a light
# analysis/viz stack into the kernel (separate from the pipeline env above).
import sys
!{sys.executable} -m pip install -q py3Dmol mdtraj rdkit pandas matplotlib numpy openmm
print("kernel viz/analysis stack ready")

In [ ]:
# Verify BOTH sides. Capture the env-python output explicitly — Colab does not
# reliably surface subprocess stdout written to an inherited file descriptor.
import subprocess
ENVPY = "/usr/local/envs/openmm-ligands/bin/python"
check = r"""
import importlib, shutil
for m in ["openmm","openff.toolkit","openmmforcefields","rdkit","parmed","mdtraj","pymol"]:
    try:
        importlib.import_module(m); print("ok   ", m)
    except Exception as e:
        print("FAIL ", m, "->", type(e).__name__, e)
for e in ["antechamber","sqm","MMPBSA.py","ante-MMPBSA.py","obabel"]:
    print(("ok   " if shutil.which(e) else "MISS "), e, "->", shutil.which(e))
"""
print("== pipeline env (openmm-ligands) ==")
r = subprocess.run([ENVPY, "-c", check], capture_output=True, text=True)
print(r.stdout, end="")
if r.stderr.strip():
    print("stderr:\n" + r.stderr)
if r.returncode != 0:
    print(f"!! env check exited {r.returncode} — env may be incomplete; re-run the env cell")

print("\n== kernel (viz/analysis) ==")
for m in ["py3Dmol","mdtraj","rdkit","pandas","matplotlib","numpy","openmm"]:
    try: __import__(m); print("ok   ", m)
    except Exception as e: print("FAIL ", m, e)
print("\nWatch the 'antechamber' line — MISS there means AM1-BCC charging will fail.")

In [ ]:
# Config: pipeline env python, script dirs (absolute), force fields, pH, platform
import os, sys
%cd /content
!mkdir -p work

ENV = "/usr/local/envs/openmm-ligands"
PY  = ENV + "/bin/python"                 # <-- every pipeline step runs with THIS python
os.environ.setdefault("PATH", "")
if ENV + "/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = ENV + "/bin:" + os.environ["PATH"]
os.environ["LIGAND_LOG_LEVEL"] = "INFO"

SR = "/content/openmm_ligands_json/scripts_running"
SP = "/content/openmm_ligands_json/scripts_postprocessing"
FF   = "amber/ff14SB.xml"           # protein force field
FFW  = "amber/tip3p_standard.xml"   # water force field
LFF  = "gaff-2.11"                  # ligand force field
PH   = 7.0

def pick_platform():
    from openmm import Platform
    names = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
    for pl in ("CUDA", "OpenCL"):
        if pl in names:
            return pl, "0"
    return "CPU", "2"

PLAT, DEV = pick_platform()
print("python:", PY)
print("force fields:", FF, "/", FFW, "/", LFF, "| pH", PH)
print("compute platform:", PLAT, "device", DEV, "| log level:", os.environ["LIGAND_LOG_LEVEL"])

## 1 · Get a protein–ligand complex

We use **3PTB** — bovine **trypsin** bound to the inhibitor **benzamidine**
(ligand `BEN`), a small, clean, classic tutorial system. Its ligand is tiny
(~9 heavy atoms), so the AM1-BCC charge fit (`antechamber`/`sqm`) that dominates
setup finishes in seconds rather than the many minutes a large drug would take.
(The structure also has a structural Ca²⁺ that the pipeline harmlessly drops —
you'll see a `skipping … (excluded)` line.)

To run **your own** structure (e.g. a Boltz / AlphaFold3 prediction like the one
this pipeline was tested on), set `COMPLEX` to an uploaded PDB and `LIG_RESNAME`
to the ligand's residue name. Note a **big** drug ligand means a **slow** `sqm`
charge fit (minutes on CPU) — it runs once and is cached in `work/ligand_cache.json`.

In [ ]:
COMPLEX     = "work/complex.pdb"
LIG_RESNAME = "BEN"        # organic ligand residue name in the PDB (benzamidine)
!wget -q https://files.rcsb.org/download/3PTB.pdb -O {COMPLEX}

# --- To upload your own complex instead, uncomment: ---
# from google.colab import files
# up = files.upload(); COMPLEX = list(up)[0]
# LIG_RESNAME = "LIG"      # <-- your ligand resname
print("complex:", COMPLEX, "| ligand resname:", LIG_RESNAME)

In [ ]:
# 3D viewer helpers (py3Dmol)
import py3Dmol, mdtraj as md, numpy as np

def view_pdb(path, ligand_resn=None, w=680, h=480):
    v = py3Dmol.view(width=w, height=h)
    v.addModel(open(path).read(), "pdb")
    v.setStyle({"cartoon": {"color": "spectrum"}})
    if ligand_resn:
        v.addStyle({"resn": ligand_resn}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.2}})
        v.addStyle({"resn": ligand_resn}, {"sphere": {"scale": 0.25}})
        v.zoomTo({"resn": ligand_resn})
    else:
        v.zoomTo()
    return v.show()

def view_sdf(path, w=540, h=400):
    v = py3Dmol.view(width=w, height=h)
    v.addModel(open(path).read(), "sdf")
    v.setStyle({"stick": {"colorscheme": "greenCarbon"}})
    v.zoomTo(); return v.show()

print("viewers ready")

In [ ]:
# The raw input: protein (cartoon) + ligand (sticks)
view_pdb(COMPLEX, ligand_resn=LIG_RESNAME)

## 2 · Split into protein + ligand

`split_ligand.py` pulls out the protein and writes one SDF per organic ligand,
re-protonating the ligand at the target **pH** with OpenBabel (`obabel -p`).

In [ ]:
!{PY} {SR}/split_ligand.py -p {COMPLEX} -o work --prefix sys \
       --ligand-list work/ligands.json --ligand-ph {PH}

import json
ligs = json.load(open("work/ligands.json"))
LIG_SDF = ligs["ligands"][0]["sdf"]
print(json.dumps(ligs, indent=2))

In [ ]:
# The extracted, protonated ligand in 3D
view_sdf(LIG_SDF)

In [ ]:
# ...and its 2D structure + SMILES (RDKit)
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
m = Chem.MolFromMolFile(LIG_SDF, removeHs=True)
print("SMILES:", Chem.MolToSmiles(m), "| formal charge:", Chem.GetFormalCharge(m))
AllChem.Compute2DCoords(m)
Draw.MolToImage(m, size=(520, 360))

In [ ]:
# The protein alone (ligand removed)
view_pdb("work/sys_protein.pdb")

## 3 · Fix the protein

Add missing atoms and **pH-aware hydrogens** (protein only — the ligand was
already protonated in step 2).

In [ ]:
!{PY} {SR}/pdb_fixer.py -p work/sys_protein.pdb -ph {PH} -o work/fixed \
       --ff {FF} --ff_water {FFW}

## 4 · Build & parameterize (the *parameterize-once* step)

Vacuum-minimize (the ligand rejoins the protein here), then **solvate and freeze
the parameters** into `solvated_system.xml`. AM1-BCC/antechamber runs **once**
and is cached; every later step just deserializes the System.

In [ ]:
# 4a. vacuum minimization — ligand joins protein, parameters cached
!{PY} {SR}/minimization.py vacuum -i work/fixed.cif --ligands_json work/ligands.json \
       --force_field {FF} --force_field_water {FFW} --ligand_force_field {LFF} \
       --cache work/ligand_cache.json --steps 1000 --platform {PLAT} --device {DEV} \
       -o work/min_vac

In [ ]:
# 4b. solvate + parameterize once  ->  solvated_system.xml (frozen) + solvated.pkl
!{PY} {SR}/system_creation.py -i work/min_vac.pkl --ligands_json work/ligands.json \
       --force_field {FF} --force_field_water {FFW} --ligand_force_field {LFF} \
       --cache work/ligand_cache.json --box_size 1.0 --ionic_strength 0.15 \
       -o work/solvated

In [ ]:
# 4c. minimize the solvated system
!{PY} {SR}/minimization.py solvated -t work/solvated.pkl -x work/solvated_system.xml \
       --steps 2000 --platform {PLAT} --device {DEV} -o work/min_sol

In [ ]:
# Visualize the SOLVATED system: protein (cartoon) + ligand (sticks) +
# the waters right around the ligand (so we don't render the whole ~30k-atom box).
from openmm.app import PDBxFile, PDBFile
cif = PDBxFile("work/solvated.cif")
with open("work/solvated.pdb", "w") as f:
    PDBFile.writeFile(cif.topology, cif.positions, f, keepIds=True)

t = md.load("work/solvated.pdb")
lig_idx = t.top.select(f"resname {LIG_RESNAME}")
wat_o   = t.top.select("water and name O")
near_o  = md.compute_neighbors(t, 0.6, lig_idx, haystack_indices=wat_o)[0]
near_res = {t.top.atom(i).residue.index for i in near_o}
wat_atoms = [a.index for a in t.top.atoms if a.residue.index in near_res]
sel = np.concatenate([t.top.select(f"protein or resname {LIG_RESNAME}"),
                      np.array(wat_atoms, dtype=int)])
t.atom_slice(sel).save_pdb("work/solvated_view.pdb")

v = py3Dmol.view(width=760, height=560)
v.addModel(open("work/solvated_view.pdb").read(), "pdb")
v.setStyle({"cartoon": {"color": "spectrum"}})
v.addStyle({"resn": LIG_RESNAME}, {"stick": {"colorscheme": "greenCarbon"}})
v.addStyle({"resn": "HOH"}, {"sphere": {"scale": 0.22}})   # nearby waters
v.zoomTo({"resn": LIG_RESNAME}); v.show()

## 5 · Equilibrate (NVT ramp → NPT) and run production MD

Step counts here are **tiny for a fast demo** — scale them up for real runs
(see the notes at the bottom).

In [ ]:
# 5a. NVT: 5K -> 300K ramp, protein backbone restrained
!{PY} {SR}/equilibration_nvt_steps.py -t work/min_sol.pkl -x work/solvated_system.xml \
       --steps 4000 --recorder 200 --time_step 0.001 \
       --platform {PLAT} --device {DEV} -o work/nvt

In [ ]:
# 5b. NPT equilibration (barostat + restraints)
!{PY} {SR}/equilibration_npt.py -t work/solvated.pkl -x work/solvated_system.xml \
       -r work/nvt.xml --apply_restraints --steps 4000 --time_step 0.002 \
       --recorder 2000 --platform {PLAT} --device {DEV} -o work/npt

In [ ]:
# 5c. production MD (no restraints)  -> 40 ps, 20 frames
!{PY} {SR}/equilibration_npt.py -t work/solvated.pkl -x work/solvated_system.xml \
       -r work/npt.xml --steps 20000 --time_step 0.002 --recorder 1000 \
       --platform {PLAT} --device {DEV} -o work/production

## 5½ · Equilibration diagnostics (NVT & NPT)

Before trusting production, check the equilibration behaved: **NVT** should ramp the
temperature to ~300 K, and **NPT** (barostat on) should settle the **density** near
~1 g/mL with the **box volume** reaching a plateau. Both steps log these to
`work/nvt.csv` / `work/npt.csv` (OpenMM `StateDataReporter`). Demo step counts are
tiny, so these curves are illustrative, not converged.

In [ ]:
# Read the NVT/NPT StateDataReporter logs and plot the equilibration observables.
import pandas as pd, matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
def col(df, key): return [c for c in df.columns if key in c][0]

nvt = pd.read_csv("work/nvt.csv")
npt = pd.read_csv("work/npt.csv")

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0,0].plot(nvt[col(nvt, "Temperature")].values, color="tab:red")
ax[0,0].set(title="NVT: temperature ramp (5K → 300K)", xlabel="record", ylabel="T (K)")
ax[0,1].plot(nvt[col(nvt, "Potential Energy")].values)
ax[0,1].set(title="NVT: potential energy", xlabel="record", ylabel="kJ/mol")
ax[1,0].plot(npt[col(npt, "Density")].values, color="tab:green")
ax[1,0].set(title="NPT: density equilibration", xlabel="record", ylabel="g/mL")
ax[1,1].plot(npt[col(npt, "Box Volume")].values, color="tab:purple")
ax[1,1].set(title="NPT: box volume", xlabel="record", ylabel="nm³")
plt.tight_layout(); plt.show()

half = max(1, len(npt)//2)
print("NVT  T: %.0f → %.0f K   |   NPT (last half)  ⟨ρ⟩=%.3f g/mL, ⟨V⟩=%.1f nm³" % (
    nvt[col(nvt,"Temperature")].iloc[0], nvt[col(nvt,"Temperature")].iloc[-1],
    npt[col(npt,"Density")].tail(half).mean(), npt[col(npt,"Box Volume")].tail(half).mean()))

## 6 · Analysis

Unwrap the periodic trajectory (keeping the ligand with its binding site), then
compute ligand RMSD and protein RMSF and plot the recorded thermodynamics.

In [ ]:
!{PY} {SP}/unwrapp.py -t work/production.dcd -to work/production.pdb -o work/unwrapped.dcd
!{PY} {SP}/ligand_rmsd.py -t work/unwrapped.dcd -to work/unwrapped_topology.pdb \
       --align_selection "protein and name CA" --ref_frame 0 -o work/ligand_rmsd.csv
!{PY} {SP}/calculate_rmsf.py -t work/unwrapped.dcd -to work/unwrapped_topology.pdb \
       --selection "protein and name CA" -o work/rmsf.csv

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

lr   = pd.read_csv("work/ligand_rmsd.csv")
rf   = pd.read_csv("work/rmsf.csv")
prod = pd.read_csv("work/production.csv")
def col(df, key): return [c for c in df.columns if key in c][0]

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ax[0,0].plot(lr["frame"], lr["mean_rmsd_nm"] * 10, "o-", ms=3)
ax[0,0].set(title="Ligand RMSD (heavy atom, protein-aligned)", xlabel="frame", ylabel="RMSD (Å)")
ax[0,1].plot(rf["residue_seq"], rf["rmsf_angstrom"])
ax[0,1].set(title="Protein Cα RMSF", xlabel="residue", ylabel="RMSF (Å)")
ax[1,0].plot(prod[col(prod, "Potential Energy")])
ax[1,0].set(title="Potential energy", xlabel="record", ylabel="kJ/mol")
ax[1,1].plot(prod[col(prod, "Temperature")], color="tab:red")
ax[1,1].set(title="Temperature", xlabel="record", ylabel="K")
plt.tight_layout(); plt.show()

In [ ]:
# Animate the ligand in the pocket over the trajectory (dry, protein-aligned)
traj = md.load("work/unwrapped.dcd", top="work/unwrapped_topology.pdb")
dry  = traj.atom_slice(traj.top.select(f"protein or resname {LIG_RESNAME}"))
dry.superpose(dry, 0, atom_indices=dry.top.select("protein and name CA"))
dry.save_pdb("work/traj_view.pdb")

v = py3Dmol.view(width=760, height=560)
v.addModelsAsFrames(open("work/traj_view.pdb").read(), "pdb")
v.setStyle({"cartoon": {"color": "lightgrey"}})
v.addStyle({"resn": LIG_RESNAME}, {"stick": {"colorscheme": "greenCarbon"}})
v.zoomTo({"resn": LIG_RESNAME})
v.animate({"loop": "forward", "interval": 300})
v.show()

## 7 · MM-GBSA binding free energy (AmberTools)

Export Amber topologies (the pipeline rebuilds an **unconstrained** System, strips
the box, and sets **mbondi2** GB radii — the three things a naive OpenMM→Amber
export gets wrong), split receptor/ligand with `ante-MMPBSA.py`, build a dry
pbc-removed trajectory, and run `MMPBSA.py` (single-trajectory, igb=5).

In [ ]:
# 7a. export Amber prmtops (dry complex: no box, mbondi2 radii)
!{PY} {SP}/export_amber.py -t work/solvated.pkl --ligands_json work/ligands.json \
       --cache work/ligand_cache.json --ligand_resnames {LIG_RESNAME} \
       -o work/mmpbsa --json work/mmpbsa.json

In [ ]:
# 7b. build the dry, pbc-removed trajectory that matches the dry prmtop
import os
os.makedirs("work/mmpbsa_run", exist_ok=True)
traj = md.load("work/unwrapped.dcd", top="work/unwrapped_topology.pdb")
dry  = traj.atom_slice(traj.top.select(f"protein or resname {LIG_RESNAME}"))
dry.superpose(dry, 0, atom_indices=dry.top.select("protein and name CA"))
dry.save_dcd("work/mmpbsa_run/dry.dcd")
NFRAMES = dry.n_frames
print("dry.dcd:", dry.n_atoms, "atoms,", NFRAMES, "frames")

In [ ]:
# 7c. write mmpbsa.in, split receptor/ligand, run MMPBSA.py
import subprocess, textwrap
rundir = "work/mmpbsa_run"
open(f"{rundir}/mmpbsa.in", "w").write(textwrap.dedent(f"""\
    Single-trajectory MM-GBSA (igb=5)
    &general
      startframe=1, endframe={NFRAMES}, interval=1, verbose=2,
    /
    &gb
      igb=5, saltcon=0.150,
    /
"""))
env = dict(os.environ, AMBERHOME=ENV)   # ante-MMPBSA/MMPBSA live in the env bin (on PATH)
subprocess.run(["ante-MMPBSA.py", "-p", "../mmpbsa_complex_dry.prmtop",
                "-c", "com.prmtop", "-r", "rec.prmtop", "-l", "lig.prmtop",
                "-s", ":WAT,HOH,Na+,Cl-,NA,CL", "-n", f":{LIG_RESNAME}"],
               cwd=rundir, env=env, check=True)
subprocess.run(["MMPBSA.py", "-O", "-i", "mmpbsa.in",
                "-cp", "../mmpbsa_complex_dry.prmtop", "-rp", "rec.prmtop", "-lp", "lig.prmtop",
                "-y", "dry.dcd", "-o", "FINAL_RESULTS.dat", "-eo", "FINAL_RESULTS.csv"],
               cwd=rundir, env=env, check=True)
print(open(f"{rundir}/FINAL_RESULTS.dat").read().split("Differences")[-1][:1200])

In [ ]:
# 7d. plot the ΔG decomposition
import re
txt   = open("work/mmpbsa_run/FINAL_RESULTS.dat").read()
delta = txt.split("Differences (Complex - Receptor - Ligand):")[1]
rows  = {}
for line in delta.splitlines():
    mm = re.match(r"^(VDWAALS|EEL|EGB|ESURF|DELTA TOTAL)\s+(-?\d+\.\d+)", line)
    if mm: rows[mm.group(1)] = float(mm.group(2))
print("ΔTOTAL (MM-GBSA binding free energy) = %.2f kcal/mol" % rows["DELTA TOTAL"])

comps = ["VDWAALS", "EEL", "EGB", "ESURF"]
plt.figure(figsize=(7, 4))
bars = plt.bar(comps, [rows[c] for c in comps],
               color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
plt.axhline(0, color="k", lw=0.7)
plt.ylabel("kcal/mol"); plt.title("MM-GBSA energy decomposition (Δ = complex − rec − lig)")
for b, c in zip(bars, comps):
    plt.text(b.get_x() + b.get_width()/2, b.get_height(),
             f"{rows[c]:.1f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

## Notes & scaling

- **Demo step counts are tiny** (fast, *not* converged). Real runs: NVT ~200k,
  NPT ~500k, production ~5M steps; MM-GBSA over ≥50 well-spaced frames.
- **MM-GBSA** here is single-trajectory GB (`igb=5`). For Poisson–Boltzmann add a
  `&pb` block. Only meaningful for a **matched AMBER-family** force field (the
  defaults used here).
- **Your own system:** set `COMPLEX` and `LIG_RESNAME` in step 1. Apo proteins:
  drop the `--ligand*` flags throughout — the pipeline is a strict superset.
- The pipeline targets AmberTools **`MMPBSA.py`**, not `gmx_MMPBSA` (which only
  ingests GROMACS files). See `README_LIGANDS.md`.
- Scale-up / at-scale runs use the JSON config + `step_runner.py` /
  `run_pipeline.py`, not these standalone calls.

## 8 · Doing it "for real": JSON config + SLURM at scale

The cells above call each script by hand — great for *seeing* the pipeline. In
practice you drive it with a **single JSON file that is both the config and the
run-state tracker**, and launch many complexes with one **SLURM** array job.

**`create_config.py`** turns a protein PDB + the `split_ligand.py` ligand list into
that JSON: shared system settings, per-step parameters (`nvt`/`npt`/`production`),
one entry per **replica**, and a full **postprocessing** block. It then prints the
exact `step_runner.py` / `postprocessing_runner.py` commands to run.

In [ ]:
# Build a real config from the 3PTB split we did above, and inspect it.
!{PY} {SR}/create_config.py \
    -p work/sys_protein.pdb --ligands_json work/ligands.json \
    -o runs/ --prefix trypsin_ben \
    --force_field {FF} --force_field_water {FFW} --ligand_force_field {LFF} \
    --ph {PH} --n_replicas 3 --box_size 1.0 \
    --nvt_steps 200000 --npt_steps 500000 --prod_steps 5000000

import json
cfg = json.load(open("runs/trypsin_ben/trypsin_ben.json"))
show = {k: cfg[k] for k in ["prefix","force_field","ligand_force_field","ligands","nvt","npt","production"]}
print(json.dumps(show, indent=2))
print("\nreplicas:", [r["id"] for r in cfg["replicas"]],
      "\npostprocessing steps:", list(cfg["postprocessing"]))

### Running from the JSON

Three drivers read that JSON (and write results back into it, so reruns skip
finished steps):

```bash
# whole shared pipeline end-to-end (pdb_fixer -> production):
python scripts_running/run_pipeline.py runs/trypsin_ben/trypsin_ben.json

# or one step at a time (per replica for nvt..production):
python scripts_running/step_runner.py --config <cfg.json> --step system_creation
python scripts_running/step_runner.py --config <cfg.json> --step production --replica 0

# postprocessing (per replica), incl. ligand_rmsd, export_amber, prepare_mmpbsa:
python scripts_postprocessing/postprocessing_runner.py --config <cfg.json> --step ligand_rmsd --replica 0
```

Apo proteins: just omit `--ligands_json` / `--ligand` when creating the config —
the pipeline is a strict superset.

In [ ]:
# The at-scale driver: one SLURM array task per complex (split -> config ->
# shared steps -> replicas -> postprocessing). Here is the real submit script:
!cat openmm_ligands_json/sh_scripts/ligand_submit.sh

### Submitting on a cluster

Edit the top of `sh_scripts/ligand_submit.sh` for your data — `COMPLEXES=(...)`
(one PDB basename per array index), `INPUT_DIR`, `OUTPUT_DIR`, `SCRIPT_DIR`, the
`#SBATCH --array=0-N`, and resources (`--gpus`, `--mem`, `--cpus-per-task`). Then:

```bash
sbatch sh_scripts/ligand_submit.sh                       # array of complexes, 1 GPU each
sbatch --export=ALL,HMR=1,PH=7.4 sh_scripts/ligand_submit.sh   # HMR (4 fs) + set pH
```

Each task splits its complex, freezes the discovered ligand list into a JSON,
parameterizes the ligand **once** (`system_creation`), runs all replicas, and
runs ligand-aware postprocessing. `--skip_if_done` makes requeued jobs resume.
(This is a login-node `sbatch` step — it won't run inside Colab, which has no
scheduler; the JSON-driven single-machine path above is the Colab-runnable form.)